# 11 Search: cost-guided search for cheaper correct queries (A100)

The first pool has a query at least 2x cheaper than gold for only about 6% of train questions, so training had little to learn from. This notebook searches harder. Each round, every question without a 2x cheaper query gets one prompt with its cheapest known correct query, that query's cost in SQLite steps and its query plan, and the earlier attempts with what went wrong (different result, error, not cheaper, matches only by chance). The model answers 4 times. Every answer is executed and kept only if it returns gold's result, passes the arbitrary-row check and uses no constant that appears nowhere in the question, evidence or gold. 3 rounds.

**Pilot** (`LIMIT = 300`): the same 300 random train questions (never from the validation databases), searched twice:
1. with **Qwen2.5-Coder-32B** (4-bit AWQ), the stronger searcher;
2. with our **7B** model in the same loop, to separate the gain of the loop from the gain of the bigger model.

The last cell compares both with the first pool on the same questions. Go-on rule: the share with a 2x cheaper query at least doubles (12% or more), or the geometric mean cost of the cheapest known query falls to 0.75 of gold or below.

**Runtime:** A100. Staging the train databases takes a few minutes, downloading the 32B model (19 GB) a few more; the searches themselves should take well under an hour together. **Resumable:** after a disconnect, run the setup and config cells again, then the cell that was running (answers and executions are saved as they finish).

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
LIMIT = 300                 # pilot; the full search runs later with LIMIT = None and a new name
NAME = "search_pilot"       # runs <NAME>_32b and <NAME>_7b in 586/effsql/runs/
LIM = f"--limit {LIMIT}" if LIMIT else ""
R = f"{DRIVE_ROOT}/effsql/runs"
import os
assert os.path.exists(f"{R}/pool_v1/pool_train.parquet"), "the search starts from pool_v1"
print("CPU cores:", os.cpu_count())
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Stage the train databases (30 GB, a few minutes; skipped when already staged)

In [ ]:
!effsql stage --split train
!effsql schema-cache --split train

### 2. Search with the 32B model

In [ ]:
!python -m effsql.gen.search --run {NAME}_32b --model 32b {LIM}

### 3. Search with the 7B model (same questions, same loop)

In [ ]:
!python -m effsql.gen.search --run {NAME}_7b --model 7b {LIM}

### 4. Compare with the first pool on the same questions

`pct_2x`: share of questions with a usable query at least 2x cheaper than gold; `pct_10pct`: at least 10% cheaper; `geomean_rel_cost`: geometric mean cost of the cheapest usable query relative to gold (1 = gold). `start` is the first pool.

In [ ]:
import json
import pandas as pd
rows = []
for m in ("32b", "7b"):
    f = f"{R}/{NAME}_{m}/search_summary_train.json"
    if not os.path.exists(f):
        continue
    s = json.load(open(f))
    rows.append({"model": m, "round": "start (pool_v1)", **s["start"]})
    for r in s["rounds"]:
        rows.append({"model": m, **r})
    for g in s["generation"]:
        print(m, "round", g["round"], f'{g["n_active"]} questions, {g["elapsed_s"]:.0f} s, '
              f'{g["generated_tokens_per_s"] or 0:.0f} tokens/s')
pd.set_option("display.width", 200)
print(pd.DataFrame(rows).to_string(index=False))

### Done: disconnect the runtime (an idle A100 still costs units)